In [1]:
# 📚 Libraries
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm
import torch

# 📂 Paths
RESAMPLED_SCANS_DIR = "../Desktop/project - Copy/preprocessed/resampled"
DETECTION_DATA_DIR = r"D:\luna16\detection_patches"
DETECTION_DATASET_CSV ="../Desktop/project - Copy/preprocessed/detection_dataset.csv"

# Create directory if it doesn't exist
os.makedirs(DETECTION_DATA_DIR, exist_ok=True)

# 📋 Load the detection dataset (already saved as CSV)
detection_dataset = pd.read_csv(DETECTION_DATASET_CSV)

print(f"✅ Loaded {len(detection_dataset)} candidates for detection dataset creation.")


✅ Loaded 754975 candidates for detection dataset creation.


In [2]:
def world_to_voxel(world_coord, origin, spacing):
    stretched_voxel_coord = np.absolute(world_coord - origin)
    voxel_coord = stretched_voxel_coord / spacing
    return voxel_coord


 Patch Extraction Function

In [3]:
def extract_patch(scan_array, center_voxel, patch_size=64):
    """
    Extract a cubic patch centered on center_voxel with side length patch_size.
    """
    center_voxel = np.round(center_voxel).astype(int)
    
    half_size = patch_size // 2
    z, y, x = center_voxel
    
    start_z = max(z - half_size, 0)
    end_z = min(z + half_size, scan_array.shape[0])
    
    start_y = max(y - half_size, 0)
    end_y = min(y + half_size, scan_array.shape[1])
    
    start_x = max(x - half_size, 0)
    end_x = min(x + half_size, scan_array.shape[2])
    
    patch = scan_array[start_z:end_z, start_y:end_y, start_x:end_x]
    
    # Padding if necessary (for border candidates)
    padding = [
        (0, patch_size - patch.shape[0]),
        (0, patch_size - patch.shape[1]),
        (0, patch_size - patch.shape[2]),
    ]
    
    patch = np.pad(patch, padding, mode='constant', constant_values=0)
    return patch


In [5]:
pip install h5py


Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



Extract 3D patches and save


In [7]:
from ast import literal_eval
import os
import numpy as np
import torch
import SimpleITK as sitk
from tqdm import tqdm
import h5py

# 🔵 Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔵 Using device: {device}")

# 🔵 Reparse the 'center' field if necessary
if isinstance(detection_dataset.iloc[0]["center"], str):
    detection_dataset["center"] = detection_dataset["center"].apply(literal_eval)

# 🔵 Preload all scans into memory
print("🔵 Preloading scans into memory...")
scan_cache = {}

for scan_id in tqdm(detection_dataset["scan_id"].unique(), desc="Loading scans"):
    scan_path = os.path.join(RESAMPLED_SCANS_DIR, f"{scan_id}_resampled.mhd")
    if not os.path.exists(scan_path):
        continue
    scan = sitk.ReadImage(scan_path)
    scan_array = sitk.GetArrayFromImage(scan)
    origin = np.array(scan.GetOrigin())[::-1]    # reverse (z, y, x)
    spacing = np.array(scan.GetSpacing())[::-1]  # reverse (z, y, x)
    scan_cache[scan_id] = (scan_array, origin, spacing)

print(f"🔵 {len(scan_cache)} scans loaded into memory.")

# 🔵 Create output file (HDF5)
output_file = os.path.join(DETECTION_DATA_DIR, "patches.h5")

# 🔵 Initialize batch buffers
positive_patches = []
negative_patches = []
batch_size = 10000  # Save every 10,000 patches
pos_batch_idx = 0
neg_batch_idx = 0

def save_to_hdf5(patches, label, batch_idx):
    with h5py.File(output_file, 'a') as f:
        group_name = f"{'positive' if label == 1 else 'negative'}_batch_{batch_idx}"
        group = f.create_group(group_name)
        patches_array = np.array([patch for patch in patches])
        group.create_dataset('patches', data=patches_array)
    print(f"✅ Batch {batch_idx} saved to HDF5.")

# 🔵 Loop over all candidates
for idx, candidate in enumerate(tqdm(detection_dataset.itertuples(index=False), total=len(detection_dataset), desc="Extracting patches")):
    scan_id = candidate.scan_id
    center_world = np.array(candidate.center)
    strict_label = candidate.strict_label

    # Get scan from cache
    if scan_id not in scan_cache:
        continue  # Skip missing scans
    
    scan_array, origin, spacing = scan_cache[scan_id]
    
    # World → Voxel
    center_voxel = world_to_voxel(center_world, origin, spacing)
    
    # Extract patch
    patch = extract_patch(scan_array, center_voxel, patch_size=32)  # Reduced patch size to 32x32x32
    
    # Normalize patch
    patch = (patch - np.mean(patch)) / (np.std(patch) + 1e-5)
    
    # Add to correct list
    if strict_label == 1:
        positive_patches.append(patch)
        # Save if batch full
        if len(positive_patches) >= batch_size:
            save_to_hdf5(positive_patches, 1, pos_batch_idx)
            positive_patches = []  # Reset list
            pos_batch_idx += 1
    else:
        negative_patches.append(patch)
        if len(negative_patches) >= batch_size:
            save_to_hdf5(negative_patches, 0, neg_batch_idx)
            negative_patches = []  # Reset list
            neg_batch_idx += 1

# 🔵 Save any remaining patches
if positive_patches:
    save_to_hdf5(positive_patches, 1, pos_batch_idx)

if negative_patches:
    save_to_hdf5(negative_patches, 0, neg_batch_idx)

print("✅ All patches extracted and saved in batches.")


🔵 Using device: cuda
🔵 Preloading scans into memory...


Loading scans: 100%|█████████████████████████████████████████████████████████████████| 888/888 [07:51<00:00,  1.88it/s]


🔵 888 scans loaded into memory.


Extracting patches:   1%|▋                                                   | 10020/754975 [01:05<17:58:52, 11.51it/s]

✅ Batch 0 saved to HDF5.


...

✅ Batch 73 saved to HDF5.


Extracting patches: 100%|█████████████████████████████████████████████████████▊| 751500/754975 [51:13<02:54, 19.90it/s]

✅ Batch 74 saved to HDF5.


Extracting patches: 100%|█████████████████████████████████████████████████████| 754975/754975 [51:16<00:00, 245.40it/s]


✅ Batch 0 saved to HDF5.
✅ Batch 75 saved to HDF5.
✅ All patches extracted and saved in batches.


In [29]:
import random
from ast import literal_eval

# 📋 Settings
NEGATIVE_SAMPLES_TARGET = 8000  # Number of negative samples you want

# 🔵 Reparse the 'center' field if necessary
if isinstance(detection_dataset.iloc[0]["center"], str):
    detection_dataset["center"] = detection_dataset["center"].apply(literal_eval)

# 🔵 Split positives and negatives
positive_candidates = detection_dataset[detection_dataset["strict_label"] == 1]
negative_candidates = detection_dataset[detection_dataset["strict_label"] == 0]

print(f"Found {len(positive_candidates)} strict positives and {len(negative_candidates)} negatives.")

# 🔵 Sample negatives randomly
if len(negative_candidates) > NEGATIVE_SAMPLES_TARGET:
    negative_candidates = negative_candidates.sample(NEGATIVE_SAMPLES_TARGET, random_state=42)

# 🔵 Merge positive + sampled negatives
final_candidates = pd.concat([positive_candidates, negative_candidates]).reset_index(drop=True)

print(f"🔵 Total candidates to process: {len(final_candidates)}")

# 📂 Create folders
pos_dir = os.path.join(DETECTION_DATA_DIR, "positive")
neg_dir = os.path.join(DETECTION_DATA_DIR, "negative")
os.makedirs(pos_dir, exist_ok=True)
os.makedirs(neg_dir, exist_ok=True)

# 🔵 Extraction loop
for idx, candidate in tqdm(final_candidates.iterrows(), total=len(final_candidates), desc="Extracting final patches"):
    scan_id = candidate["scan_id"]
    center_world = np.array(candidate["center"])  # (z, y, x)
    strict_label = candidate["strict_label"]

    scan_path = os.path.join(RESAMPLED_SCANS_DIR, f"{scan_id}_resampled.mhd")
    
    if not os.path.exists(scan_path):
        continue  # Skip if scan missing
    
    # Load scan
    scan = sitk.ReadImage(scan_path)
    scan_array = sitk.GetArrayFromImage(scan)
    origin = np.array(scan.GetOrigin())[::-1]
    spacing = np.array(scan.GetSpacing())[::-1]
    
    # World → Voxel
    center_voxel = world_to_voxel(center_world, origin, spacing)
    
    # Extract patch
    patch = extract_patch(scan_array, center_voxel, patch_size=64)
    
    # Normalize
    patch = (patch - np.mean(patch)) / (np.std(patch) + 1e-5)
    
    # Save
    patch_tensor = torch.tensor(patch).float().unsqueeze(0)
    
    save_dir = pos_dir if strict_label == 1 else neg_dir
    patch_save_path = os.path.join(save_dir, f"{scan_id}_{idx}.pt")
    torch.save(patch_tensor, patch_save_path)


Found 1215 strict positives and 753760 negatives.
🔵 Total candidates to process: 9215


Extracting final patches: 100%|████████████████████████████████████████████████████| 9215/9215 [28:14<00:00,  5.44it/s]


DetectionDataset Class

In [31]:
from torch.utils.data import Dataset
import torch
import os
import random

class DetectionDataset(Dataset):
    def __init__(self, root_dir, mode="train", positives_ratio=0.5):
        """
        root_dir: path to detection_patches/
        mode: "train" or "val" (for now we'll use only "train")
        positives_ratio: fraction of positives in batch (if balanced)
        """
        self.root_dir = root_dir
        self.mode = mode
        self.positives_ratio = positives_ratio
        
        # Collect positive and negative file paths
        self.positive_paths = [os.path.join(root_dir, "positive", f) for f in os.listdir(os.path.join(root_dir, "positive")) if f.endswith(".pt")]
        self.negative_paths = [os.path.join(root_dir, "negative", f) for f in os.listdir(os.path.join(root_dir, "negative")) if f.endswith(".pt")]
        
        self.all_paths = self.positive_paths + self.negative_paths
        self.labels = [1]*len(self.positive_paths) + [0]*len(self.negative_paths)

        print(f"✅ Dataset initialized: {len(self.positive_paths)} positives, {len(self.negative_paths)} negatives.")

    def __len__(self):
        return len(self.all_paths)

    def __getitem__(self, idx):
        path = self.all_paths[idx]
        label = self.labels[idx]
        
        patch = torch.load(path)  # (1, 64, 64, 64)
        
        return patch, label


create dataloader

In [33]:
from torch.utils.data import DataLoader, random_split

# 📋 Parameters
BATCH_SIZE = 32  # You can change (16, 32, 64 depending on your RAM/GPU)
VAL_SPLIT = 0.2  # 20% of data for validation

# 📂 Dataset path
DETECTION_DATASET_DIR = r"D:\luna16\detection_patches"

# 📦 Create the full dataset
full_dataset = DetectionDataset(root_dir=DETECTION_DATASET_DIR)

# 📦 Split into train and validation datasets
total_size = len(full_dataset)
val_size = int(total_size * VAL_SPLIT)
train_size = total_size - val_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 📦 Create the DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,        # 🔵 Important: Shuffle at every epoch
    num_workers=0,        # Adjust depending on your computer (2 is good start)
    pin_memory=True,      # If you have GPU, helps transfer faster
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # ❌ Don't shuffle validation data
    num_workers=0,
    pin_memory=True,
)

print(f"✅ Train DataLoader ready: {len(train_loader)} batches per epoch.")
print(f"✅ Validation DataLoader ready: {len(val_loader)} batches per epoch.")


✅ Dataset initialized: 1215 positives, 8000 negatives.
✅ Train DataLoader ready: 231 batches per epoch.
✅ Validation DataLoader ready: 58 batches per epoch.
